In [ ]:
import os,sys
import re
import numpy as np
import pandas as pd
import scanpy as sc
from scanpy import AnnData
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from scipy.sparse import csr_matrix
import squidpy as sq
import anndata as ad
import mudata
import harmonypy as hm

sc.set_figure_params(figsize=(4,4))
plt.rcParams['pdf.fonttype']=42
warnings.filterwarnings("ignore")
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
sc.settings.verbosity = 3

In [ ]:
control_path='/u/home/m/mbaig/project-cluo/cosmx/spatial-data/20251130_MB_basal_3/'
dataDir_spatial = '/u/home/m/mbaig/project-cluo/cosmx/analyses/20251130_MB_basal_3/processed_data/'

In [ ]:
#loading in the raw data matrix in chunks
#slide5

# Initialize an empty list to store chunks
chunks = []

# Define the chunk size
chunksize = 100000

# Read the CSV file in chunks
for chunk in pd.read_csv(control_path + 'MBbrain6Kbasalslide5redo/MBbrain6Kbasalslide5redo_exprMat_file.csv.gz', chunksize=chunksize):
    chunks.append(chunk)

# Concatenate all chunks into a single DataFrame
df1 = pd.concat(chunks, ignore_index=True)

# Display the DataFrame
print(df1.head())

In [ ]:
meta=pd.read_csv(control_path+'MBbrain6Kbasalslide5redo/MBbrain6Kbasalslide5redo_metadata_file.csv.gz')
meta=meta[meta.columns[3:]].set_index('cell')
meta

In [ ]:
df = df1

In [ ]:
# create row names for "cell" in same format as what's used in cell metadata file
# the new cell_ID has pattern as 'c_[slideID]_[cell_ID]', where slideID = 1 in this example dataset
df["new_cell_ID"] =  df.apply(lambda row: f"c_1_{row.fov}_{row.cell_ID}", axis = 1)
df.set_index("new_cell_ID", inplace=True)

# drop annotation columns
df.drop(columns=["fov", "cell_ID"], inplace=True)

# focus on RNA probes while drop control probes
dummy=re.compile(r'Negative|SystemControl', flags=re.IGNORECASE)
chosen_probes = [col for col in list(df.columns) if not dummy.search(col)]

# "cell" names for ordering downstream
chosen_cells = df.index

# convert to a sparse matrix
raw = csr_matrix(df[chosen_probes].astype(pd.SparseDtype("float64",0)).sparse.to_coo())

# delete unused variable to free up memory
del df

In [ ]:
# read in cell meta data file
#cell_meta = pd.read_csv(os.path.join(dataDir, 'Pancreas_metadata_file.csv'))
cell_meta=pd.read_csv(control_path+'MBbrain6Kbasalslide5redo/MBbrain6Kbasalslide5redo_metadata_file.csv.gz')

# use "cell" column as row names to match with raw expression matrix
# for convenience, the "cell" column is kept with the meta data
cell_meta.set_index("cell", inplace=True, drop = False)

# extract spatial coordinates of cells, use the global coordinates
coords = cell_meta[["CenterX_global_px","CenterY_global_px"]]

# use "x" and "y" as column names
coords = coords.rename(columns={"CenterX_global_px":"x",
                                "CenterY_global_px": "y"})

# reorder cells to be in same row order as raw expression matrix
coords = coords.reindex(index = chosen_cells)

# convert px coordinates to micrometer
pixel_size = 0.12028
coords = coords.mul(pixel_size)

In [ ]:
adata = ad.AnnData(
    X = raw,
    obs = cell_meta,
    # row names should be the same as gene names
    var = pd.DataFrame(
        list(chosen_probes),
        columns = ["gene"],
        index = chosen_probes))

# add spatial coordinates
adata.obsm['spatial'] = coords.to_numpy()

# you can add name of slide or original file as one of unstructured annotation
adata.uns['name'] = "MBbrain6Kbasalslide5"

# convert string columns to categorical data in `obs`
adata.strings_to_categoricals()

In [ ]:
plt.rcParams['figure.figsize']=(10,14)
sns.scatterplot(data=adata.obs,x='CenterX_global_px', y='CenterY_global_px', s=2, edgecolor=None)  # s parameter sets dot size, edgecolor=None removes outline
plt.show()

In [ ]:
adata

In [ ]:
import numpy as np

# Specify the gene(s) of interest
gene_of_interest = 'STMN2'  # Replace with the name of the gene

# Ensure the gene is in the variable names
if gene_of_interest in adata.var.index:
    # Extract counts for the specific gene
    gene_counts = adata[:, gene_of_interest].X

    # Calculate the average count
    average_count = np.mean(gene_counts)

    print(f"The average count for {gene_of_interest} is {average_count}")
else:
    print(f"The gene {gene_of_interest} is not present in the dataset.")

In [ ]:
dataDir_spatial = '/u/home/m/mbaig/project-cluo/cosmx/analyses/20251130_MB_basal_3/processed_data/20251130'

In [ ]:
adata.write(os.path.join(dataDir_spatial, "MBbrain6Kbasalslide5_20251130_raw1.h5ad"), compression="gzip")

In [ ]:
#loading in the raw data matrix in chunks
#slide6

# Initialize an empty list to store chunks
chunks = []

# Define the chunk size
chunksize = 100000

# Read the CSV file in chunks
for chunk in pd.read_csv(control_path + 'MBbrain6Kbasalslide6redo/MBbrain6Kbasalslide6redo_exprMat_file.csv.gz', chunksize=chunksize):
    chunks.append(chunk)

# Concatenate all chunks into a single DataFrame
df1 = pd.concat(chunks, ignore_index=True)

# Display the DataFrame
print(df1.head())

In [ ]:
meta=pd.read_csv(control_path+'MBbrain6Kbasalslide6redo/MBbrain6Kbasalslide6redo_metadata_file.csv.gz')
meta=meta[meta.columns[3:]].set_index('cell')
meta

In [ ]:
# read in cell meta data file
#cell_meta = pd.read_csv(os.path.join(dataDir, 'Pancreas_metadata_file.csv'))
cell_meta=pd.read_csv(control_path+'MBbrain6Kbasalslide6redo/MBbrain6Kbasalslide6redo_metadata_file.csv.gz')

# use "cell" column as row names to match with raw expression matrix
# for convenience, the "cell" column is kept with the meta data
cell_meta.set_index("cell", inplace=True, drop = False)

# extract spatial coordinates of cells, use the global coordinates
coords = cell_meta[["CenterX_global_px","CenterY_global_px"]]

# use "x" and "y" as column names
coords = coords.rename(columns={"CenterX_global_px":"x",
                                "CenterY_global_px": "y"})

# reorder cells to be in same row order as raw expression matrix
coords = coords.reindex(index = chosen_cells)

# convert px coordinates to micrometer
pixel_size = 0.12028
coords = coords.mul(pixel_size)

In [ ]:
adata = ad.AnnData(
    X = raw,
    obs = cell_meta,
    # row names should be the same as gene names
    var = pd.DataFrame(
        list(chosen_probes),
        columns = ["gene"],
        index = chosen_probes))

# add spatial coordinates
adata.obsm['spatial'] = coords.to_numpy()

# you can add name of slide or original file as one of unstructured annotation
adata.uns['name'] = "MBbrain6Kbasalslide6"

# convert string columns to categorical data in `obs`
adata.strings_to_categoricals()

In [ ]:
adata.write(os.path.join(dataDir_spatial, "MBbrain6Kbasalslide6_20251130_raw1.h5ad"), compression="gzip")

In [ ]:
control_path='/u/home/m/mbaig/project-cluo/cosmx/spatial-data/20251130_MB_basal_3/'
dataDir_spatial = '/u/home/m/mbaig/project-cluo/cosmx/analyses/20251130_MB_basal_3/processed_data/20251130'

In [ ]:
#slide5
adata1 = sc.read(os.path.join(dataDir_spatial, "MBbrain6Kbasalslide5_20251130_raw1.h5ad"))
adata1

In [ ]:
plt.rcParams['figure.figsize']=(10,14)
sns.scatterplot(data=adata1.obs,x='CenterX_global_px', y='CenterY_global_px', s=2, edgecolor=None)  # s parameter sets dot size, edgecolor=None removes outline
plt.show()

In [ ]:
#no filtering for now

adata1.layers["counts"] = adata1.X.copy()
sc.pp.normalize_total(adata1, inplace=True)
sc.pp.log1p(adata1)
sc.pp.pca(adata1)
sc.pp.neighbors(adata1)

In [ ]:
sc.tl.umap(adata1)
sc.tl.leiden(adata1)

In [ ]:
sc.pl.umap(adata1, color=['leiden'],palette='tab20', size=4)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set the figure size
plt.rcParams['figure.figsize'] = (10, 14)

# Create the scatter plot
sns.scatterplot(
    data=adata1.obs,
    x='CenterX_global_px',
    y='CenterY_global_px',
    hue='leiden',  # Color by Leiden clusters
    palette='tab20',  # Choose a color palette
    s=1,  # Set dot size
    edgecolor=None  # Remove outline
)

# Show the plot
plt.show()

In [ ]:
import scanpy as sc

# 1. PCA + neighbors (only if not already done)
#sc.pp.pca(adata1, n_comps=50, svd_solver='arpack')
#sc.pp.neighbors(adata1, n_neighbors=30, n_pcs=40)

# 2. Multi-resolution Leiden - correct syntax for Scanpy ≥1.9.6 + leidenalg ≥0.10
resolutions = [0.25, 0.5, 1.0, 2.0]

for res in resolutions:
    print(f"Running Leiden at resolution {res} ...")

    sc.tl.leiden(
        adata1,
        resolution=res,
        key_added=f"leiden_res_{res}",      #  leiden_res_0.25, leiden_res_0.5, etc.
        random_state=42,                    # for perfect reproducibility
        n_iterations=2                      # 2 is almost always enough
        # flavor and directed are now ignored - the default is igraph + undirected
    )

    n_clusters = adata1.obs[f"leiden_res_{res}"].nunique()
    print(f"    {n_clusters} clusters")

# 3. Make resolution 1.0 the default column
adata1.obs['leiden'] = adata1.obs['leiden_res_1.0'].astype('category')

# 4. Confirm success
print("\nSuccess! These columns are now in adata1.obs:")
print([c for c in adata1.obs.columns if c.startswith('leiden_res_')])
print("Default column: 'leiden'")

# Quick test plot
sc.pl.umap(adata1, color='leiden_res_0.5', legend_loc='on data', size=10)

In [ ]:
sc.pl.umap(adata1, color='leiden_res_0.25', legend_loc='on data', size=10)
sc.pl.umap(adata1, color='leiden_res_0.5', legend_loc='on data', size=10)
sc.pl.umap(adata1, color='leiden_res_1.0', legend_loc='on data', size=10)
sc.pl.umap(adata1, color='leiden_res_2.0', legend_loc='on data', size=10)

In [ ]:
adata1

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set the figure size
plt.rcParams['figure.figsize'] = (10, 14)

# Create the scatter plot
sns.scatterplot(
    data=adata1.obs,
    x='CenterX_global_px',
    y='CenterY_global_px',
    hue='leiden_res_0.25',  # Color by Leiden clusters
    palette='tab20',  # Choose a color palette
    s=1,  # Set dot size
    edgecolor=None  # Remove outline
)

# Show the plot
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set the figure size
plt.rcParams['figure.figsize'] = (10, 14)

# Create the scatter plot
sns.scatterplot(
    data=adata1.obs,
    x='CenterX_global_px',
    y='CenterY_global_px',
    hue='leiden_res_0.5',  # Color by Leiden clusters
    palette='tab20',  # Choose a color palette
    s=1,  # Set dot size
    edgecolor=None  # Remove outline
)

# Show the plot
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set the figure size
plt.rcParams['figure.figsize'] = (10, 14)

# Create the scatter plot
sns.scatterplot(
    data=adata1.obs,
    x='CenterX_global_px',
    y='CenterY_global_px',
    hue='leiden_res_1.0',  # Color by Leiden clusters
    palette='tab20',  # Choose a color palette
    s=1,  # Set dot size
    edgecolor=None  # Remove outline
)

# Show the plot
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set the figure size
plt.rcParams['figure.figsize'] = (10, 14)

# Create the scatter plot
sns.scatterplot(
    data=adata1.obs,
    x='CenterX_global_px',
    y='CenterY_global_px',
    hue='leiden_res_2.0',  # Color by Leiden clusters
    palette='tab20',  # Choose a color palette
    s=1,  # Set dot size
    edgecolor=None  # Remove outline
)

# Show the plot
plt.show()

In [ ]:
resolutions = [4.0, 8.0]

for res in resolutions:
    print(f"Running Leiden at resolution {res} ...")

    sc.tl.leiden(
        adata1,
        resolution=res,
        key_added=f"leiden_res_{res}",      #  leiden_res_0.25, leiden_res_0.5, etc.
        random_state=42,                    # for perfect reproducibility
        n_iterations=2                      # 2 is almost always enough
        # flavor and directed are now ignored - the default is igraph + undirected
    )

    n_clusters = adata1.obs[f"leiden_res_{res}"].nunique()
    print(f"    {n_clusters} clusters")

# 3. Make resolution 1.0 the default column
adata1.obs['leiden'] = adata1.obs['leiden_res_1.0'].astype('category')

# 4. Confirm success
print("\nSuccess! These columns are now in adata1.obs:")
print([c for c in adata1.obs.columns if c.startswith('leiden_res_')])
print("Default column: 'leiden'")

# Quick test plot
sc.pl.umap(adata1, color='leiden_res_8.0', legend_loc='on data', size=10)

In [ ]:
sc.pl.umap(adata1, color='leiden_res_4.0', legend_loc='on data', size=10)

In [ ]:
#now plotting back on spatial image

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Extract spatial and UMAP coordinates
spatial_coords = adata1.obs[['CenterX_global_px', 'CenterY_global_px']]
umap_coords = adata1.obsm['X_umap']
leiden_clusters = adata1.obs['leiden'].astype(str)  # Convert Leiden clusters to string for mapping

# Convert Leiden clusters to integers for proper numerical sorting
leiden_clusters_int = leiden_clusters.astype(int)
sorted_clusters = np.sort(leiden_clusters_int.unique())  # Sort numerically

# Define a consistent color palette
palette = sns.color_palette('tab20', len(sorted_clusters))
cluster_colors = {str(cluster): palette[i] for i, cluster in enumerate(sorted_clusters)}

# Create figure and subplots
fig, axes = plt.subplots(1, 2, figsize=(18, 9), gridspec_kw={'width_ratios': [1, 1]})

# Spatial plot
sns.scatterplot(x=spatial_coords['CenterX_global_px'], y=spatial_coords['CenterY_global_px'],
                hue=leiden_clusters, palette=cluster_colors, s=2, edgecolor=None, ax=axes[0])
axes[0].set_title('Leiden Clusters on Spatial Image')
axes[0].set_xlabel('CenterX (px)')
axes[0].set_ylabel('CenterY (px)')
axes[0].legend_.remove()  # Remove individual legend

# UMAP plot
sns.scatterplot(x=umap_coords[:, 0], y=umap_coords[:, 1],
                hue=leiden_clusters, palette=cluster_colors, s=5, edgecolor=None, ax=axes[1])
axes[1].set_title('Leiden Clusters on UMAP')
axes[1].set_xlabel('UMAP1')
axes[1].set_ylabel('UMAP2')
axes[1].legend_.remove()  # Remove individual legend

# Create a single shared legend closer to the plots
handles = [plt.Line2D([0], [0], marker='o', color=cluster_colors[str(cluster)], linestyle='', markersize=8)
           for cluster in sorted_clusters]
labels = [str(cluster) for cluster in sorted_clusters]
fig.legend(handles, labels, title="Leiden Clusters", bbox_to_anchor=(1, 0.5), loc='center left')

plt.tight_layout(rect=[0, 0, 0.85, 1])  # Adjust layout to fit legend closer to the plots
plt.show()

In [ ]:
adata1.write(os.path.join(dataDir_spatial, "MBbrain6Kbasalslide5_unfiltered_normalized1.h5ad"), compression="gzip")

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

# 1. Compute marker genes for leiden_res_0.25 (11 clusters)
print("Computing marker genes for leiden_res_0.25 (this takes ~1-3 min on 400k cells)...")

sc.tl.rank_genes_groups(
    adata1,
    groupby='leiden_res_0.25',
    method='wilcoxon',          # best for spatial transcriptomics
    use_raw=False,              # False if you already normalized/log-transformed
    n_genes=100,                # compute top 100 so we can safely take top 30 later
    key_added='markers_leiden_res_0.25'
)

# 2. Plot: Top 30 marker genes across the 11 clusters (dotplot)
sc.pl.rank_genes_groups_dotplot(
    adata1,
    n_genes=30,                              # top 30 per cluster
    key='markers_leiden_res_0.25',
    groupby='leiden_res_0.25',
    standard_scale='var',                    # normalize expression per gene (0-1)
    cmap='Reds',
    dot_min=0.1,
    dot_max=0.6,
    figsize=(14, 9),
    title='Top 30 marker genes per cluster (Leiden resolution 0.25)',
    save=False
)

# 3. (Optional) Also show as heatmap - very popular in CosMx papers
sc.pl.rank_genes_groups_heatmap(
    adata1,
    n_genes=30,
    key='markers_leiden_res_0.25',
    groupby='leiden_res_0.25',
    standard_scale='var',
    cmap='viridis',
    figsize=(16, 10),
    show_gene_labels=True,
    var_group_rotation=90,
    dendrogram=True,
    save=False
)

In [ ]:
import scanpy as sc
import pandas as pd

# 1. Re-compute markers (wilcoxon, fast and reliable)
sc.tl.rank_genes_groups(
    adata1,
    groupby='leiden_res_0.25',
    method='wilcoxon',
    use_raw=False,
    n_genes=50,  # enough to pick from
    key_added='markers_clean'
)

# 2. Extract only the TOP 5 genes per cluster (by p-value + logFC)
df = sc.get.rank_genes_groups_df(adata1, key='markers_clean', group=None)

# Keep only significantly enriched genes
df = df[df['pvals_adj'] < 0.05]

# Get top 5 per cluster (best combo of significance + fold-change)
top_genes_per_cluster = (
    df.loc[df.groupby('group')['logfoldchanges'].nlargest(5).index.get_level_values(1)]
      .groupby('group')['names']
      .apply(list)
)

# Union of all top 5  usually 30-50 genes max
selected_genes = sorted(set(sum(top_genes_per_cluster.tolist(), [])))

print(f"Plotting only {len(selected_genes)} highly specific marker genes (top 5 per cluster)")

# 3. Beautiful, clean dotplot (this is the one you want for papers)
sc.pl.dotplot(
    adata1,
    var_names=selected_genes,
    groupby='leiden_res_0.25',
    cmap='Reds',
    dot_min=0.15,
    dot_max_size=40,
    standard_scale='var',           # color: normalized expression 0-1
    figsize=(12, 8),
    title='Top marker genes per cluster (Leiden res 0.25)',
    save=False,
    show=True
)

# 4. Even cleaner alternative: compact heatmap (my favorite)
sc.pl.heatmap(
    adata1,
    var_names=selected_genes,
    groupby='leiden_res_0.25',
    cmap='viridis',
    standard_scale='var',
    figsize=(11, 9),
    var_group_rotation=45,
    show_gene_labels=True,
    dendrogram=False
)

In [ ]:
adata1.obs['unassignedTranscripts']

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. Compute mean unassigned fraction per FOV (same as before)
fov_summary = (
    adata1.obs
    .groupby('fov')['unassignedTranscripts']
    .mean()
    .reset_index()
    .rename(columns={'unassignedTranscripts': 'mean_unassigned_fraction'})
)

# Add the FOV center coordinates (CosMx stores global pixel positions)
fov_centers = (
    adata1.obs
    .groupby('fov')[['CenterX_global_px', 'CenterY_global_px']]
    .mean()  # center of each FOV
    .reset_index()
)

# Merge summary with spatial centers
fov_summary = fov_summary.merge(fov_centers, on='fov')

print(f"Plotting {len(fov_summary)} FOVs colored by unassigned transcript fraction")

# 2. Spatial plot: FOVs as semi-transparent squares, colored by value
plt.figure(figsize=(14, 12))

# Use a nice sequential colormap (blue = low, red = high unassigned = bad)
cmap = 'RdYlBu_r'  # red = high unassigned (bad), blue = low (good)

scatter = plt.scatter(
    fov_summary['CenterX_global_px'],
    fov_summary['CenterY_global_px'],
    c=fov_summary['mean_unassigned_fraction'],
    cmap=cmap,
    s=3800,                    # size of each FOV square (tuned for CosMx)
    edgecolor='black',
    linewidth=1.2,
    alpha=0.85,
    marker='s'                 # square marker = real FOV shape
)

# Add colorbar
cbar = plt.colorbar(scatter, shrink=0.6)
cbar.set_label('Mean Fraction of Unassigned Transcripts per FOV', fontsize=14, labelpad=15)
cbar.ax.tick_params(labelsize=12)

# Labels and title
plt.title('Spatial Distribution of Unassigned Transcripts Across FOVs\n'
          '(Red = high unassigned, Blue = low/clean)',
          fontsize=18, fontweight='bold', pad=20)
plt.xlabel('X position (global pixels)', fontsize=14)
plt.ylabel('Y position (global pixels)', fontsize=14)

# Optional: invert Y axis if your tissue appears upside-down (common in CosMx)
plt.gca().invert_yaxis()

# Clean look
plt.axis('equal')  # keeps FOVs as perfect squares
sns.despine(left=True, bottom=True)
plt.tick_params(bottom=False, left=False, labelbottom=False, labelleft=False)  # hide ticks

plt.tight_layout()
plt.show()

# 3. Bonus: Save high-res version
# plt.savefig('FOV_unassigned_transcripts_spatial.png', dpi=300, bbox_inches='tight')
# plt.savefig('FOV_unassigned_transcripts_spatial.pdf', bbox_inches='tight')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Get one value + center per FOV
fov_summary = (
    adata1.obs
    .groupby('fov')[['unassignedTranscripts', 'CenterX_global_px', 'CenterY_global_px']]
    .mean()
    .reset_index()
)

# 2. THIS IS THE ONLY LINE THAT MATTERS FOR CORRECT ORIENTATION
fov_summary['y_plot'] = fov_summary['CenterY_global_px'].max() - fov_summary['CenterY_global_px']
#  subtracts every Y from the maximum Y  perfectly flips the tissue vertically

# 3. Plot
plt.figure(figsize=(15, 13))

plt.scatter(
    fov_summary['CenterX_global_px'],
    fov_summary['y_plot'],                              #  flipped Y
    c=fov_summary['unassignedTranscripts'],
    cmap='RdYlBu_r',
    s=750,
    edgecolor='black',
    linewidth=1,
    alpha=0.88,
    marker='s'
)

plt.colorbar(label='Mean Fraction Unassigned Transcripts', shrink=0.7)
plt.title('Spatial QC - Unassigned Transcripts per FOV\n(now correctly oriented like Loupe Browser)',
          fontsize=18, fontweight='bold', pad=20)
plt.axis('equal')
plt.xticks([])
plt.yticks([])
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.show()

In [ ]:
#slide5 unfiltered normalized
adata1 = sc.read(os.path.join(dataDir_spatial, "MBbrain6Kbasalslide5_unfiltered_normalized1.h5ad"))
adata1

In [ ]:
#leiden default res

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Extract spatial and UMAP coordinates
spatial_coords = adata1.obs[['CenterX_global_px', 'CenterY_global_px']]
umap_coords = adata1.obsm['X_umap']
leiden_clusters = adata1.obs['leiden'].astype(str)  # Convert Leiden clusters to string for mapping

# Convert Leiden clusters to integers for proper numerical sorting
leiden_clusters_int = leiden_clusters.astype(int)
sorted_clusters = np.sort(leiden_clusters_int.unique())  # Sort numerically

# Define a consistent color palette
palette = sns.color_palette('tab20', len(sorted_clusters))
cluster_colors = {str(cluster): palette[i] for i, cluster in enumerate(sorted_clusters)}

# Create figure and subplots
fig, axes = plt.subplots(1, 2, figsize=(18, 9), gridspec_kw={'width_ratios': [1, 1]})

# Spatial plot
sns.scatterplot(x=spatial_coords['CenterX_global_px'], y=spatial_coords['CenterY_global_px'],
                hue=leiden_clusters, palette=cluster_colors, s=2, edgecolor=None, ax=axes[0])
axes[0].set_title('Leiden Clusters on Spatial Image')
axes[0].set_xlabel('CenterX (px)')
axes[0].set_ylabel('CenterY (px)')
axes[0].legend_.remove()  # Remove individual legend

# UMAP plot
sns.scatterplot(x=umap_coords[:, 0], y=umap_coords[:, 1],
                hue=leiden_clusters, palette=cluster_colors, s=5, edgecolor=None, ax=axes[1])
axes[1].set_title('Leiden Clusters on UMAP')
axes[1].set_xlabel('UMAP1')
axes[1].set_ylabel('UMAP2')
axes[1].legend_.remove()  # Remove individual legend

# Create a single shared legend closer to the plots
handles = [plt.Line2D([0], [0], marker='o', color=cluster_colors[str(cluster)], linestyle='', markersize=8)
           for cluster in sorted_clusters]
labels = [str(cluster) for cluster in sorted_clusters]
fig.legend(handles, labels, title="Leiden Clusters", bbox_to_anchor=(1, 0.5), loc='center left')

plt.tight_layout(rect=[0, 0, 0.85, 1])  # Adjust layout to fit legend closer to the plots
plt.show()

In [ ]:
#leiden_res_0.25

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Extract spatial and UMAP coordinates
spatial_coords = adata1.obs[['CenterX_global_px', 'CenterY_global_px']]
umap_coords = adata1.obsm['X_umap']
leiden_clusters = adata1.obs['leiden_res_0.25'].astype(str)  # Convert Leiden clusters to string for mapping

# Convert Leiden clusters to integers for proper numerical sorting
leiden_clusters_int = leiden_clusters.astype(int)
sorted_clusters = np.sort(leiden_clusters_int.unique())  # Sort numerically

# Define a consistent color palette
palette = sns.color_palette('tab20', len(sorted_clusters))
cluster_colors = {str(cluster): palette[i] for i, cluster in enumerate(sorted_clusters)}

# Create figure and subplots
fig, axes = plt.subplots(1, 2, figsize=(18, 9), gridspec_kw={'width_ratios': [1, 1]})

# Spatial plot
sns.scatterplot(x=spatial_coords['CenterX_global_px'], y=spatial_coords['CenterY_global_px'],
                hue=leiden_clusters, palette=cluster_colors, s=2, edgecolor=None, ax=axes[0])
axes[0].set_title('Leiden Clusters on Spatial Image')
axes[0].set_xlabel('CenterX (px)')
axes[0].set_ylabel('CenterY (px)')
axes[0].legend_.remove()  # Remove individual legend

# UMAP plot
sns.scatterplot(x=umap_coords[:, 0], y=umap_coords[:, 1],
                hue=leiden_clusters, palette=cluster_colors, s=5, edgecolor=None, ax=axes[1])
axes[1].set_title('Leiden Clusters on UMAP')
axes[1].set_xlabel('UMAP1')
axes[1].set_ylabel('UMAP2')
axes[1].legend_.remove()  # Remove individual legend

# Create a single shared legend closer to the plots
handles = [plt.Line2D([0], [0], marker='o', color=cluster_colors[str(cluster)], linestyle='', markersize=8)
           for cluster in sorted_clusters]
labels = [str(cluster) for cluster in sorted_clusters]
fig.legend(handles, labels, title="Leiden Clusters", bbox_to_anchor=(1, 0.5), loc='center left')

plt.tight_layout(rect=[0, 0, 0.85, 1])  # Adjust layout to fit legend closer to the plots
plt.show()

#no filtering

In [ ]:
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt

# Extract UMI counts
umi_counts = adata1.obs['nCount_RNA']

umi_counts.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])

In [ ]:
#will set filter thresholds to keep only cells with UMIs between:
#200-7500

adata1_filtered_200 = adata1[(adata1.obs['nCount_RNA'] > 200) & (adata1.obs['nCount_RNA'] < 7500)].copy()
adata1_filtered_200

In [ ]:
#leiden_res_0.25

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Extract spatial and UMAP coordinates
spatial_coords = adata1_filtered_200.obs[['CenterX_global_px', 'CenterY_global_px']]
umap_coords = adata1_filtered_200.obsm['X_umap']
leiden_clusters = adata1_filtered_200.obs['leiden_res_0.25'].astype(str)  # Convert Leiden clusters to string for mapping

# Convert Leiden clusters to integers for proper numerical sorting
leiden_clusters_int = leiden_clusters.astype(int)
sorted_clusters = np.sort(leiden_clusters_int.unique())  # Sort numerically

# Define a consistent color palette
palette = sns.color_palette('tab20', len(sorted_clusters))
cluster_colors = {str(cluster): palette[i] for i, cluster in enumerate(sorted_clusters)}

# Create figure and subplots
fig, axes = plt.subplots(1, 2, figsize=(18, 9), gridspec_kw={'width_ratios': [1, 1]})

# Spatial plot
sns.scatterplot(x=spatial_coords['CenterX_global_px'], y=spatial_coords['CenterY_global_px'],
                hue=leiden_clusters, palette=cluster_colors, s=2, edgecolor=None, ax=axes[0])
axes[0].set_title('Leiden Clusters on Spatial Image')
axes[0].set_xlabel('CenterX (px)')
axes[0].set_ylabel('CenterY (px)')
axes[0].legend_.remove()  # Remove individual legend

# UMAP plot
sns.scatterplot(x=umap_coords[:, 0], y=umap_coords[:, 1],
                hue=leiden_clusters, palette=cluster_colors, s=5, edgecolor=None, ax=axes[1])
axes[1].set_title('Leiden Clusters on UMAP')
axes[1].set_xlabel('UMAP1')
axes[1].set_ylabel('UMAP2')
axes[1].legend_.remove()  # Remove individual legend

# Create a single shared legend closer to the plots
handles = [plt.Line2D([0], [0], marker='o', color=cluster_colors[str(cluster)], linestyle='', markersize=8)
           for cluster in sorted_clusters]
labels = [str(cluster) for cluster in sorted_clusters]
fig.legend(handles, labels, title="Leiden Clusters", bbox_to_anchor=(1, 0.5), loc='center left')

plt.tight_layout(rect=[0, 0, 0.85, 1])  # Adjust layout to fit legend closer to the plots
plt.show()
adata1_filtered

#200 minimum umi cutoff

In [ ]:
#will set filter thresholds to keep only cells with UMIs between:
#300-7500

adata1_filtered_300 = adata1[(adata1.obs['nCount_RNA'] > 300) & (adata1.obs['nCount_RNA'] < 7500)].copy()
adata1_filtered_300

In [ ]:
#leiden_res_0.25

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Extract spatial and UMAP coordinates
spatial_coords = adata1_filtered_300.obs[['CenterX_global_px', 'CenterY_global_px']]
umap_coords = adata1_filtered_300.obsm['X_umap']
leiden_clusters = adata1_filtered_300.obs['leiden_res_0.25'].astype(str)  # Convert Leiden clusters to string for mapping

# Convert Leiden clusters to integers for proper numerical sorting
leiden_clusters_int = leiden_clusters.astype(int)
sorted_clusters = np.sort(leiden_clusters_int.unique())  # Sort numerically

# Define a consistent color palette
palette = sns.color_palette('tab20', len(sorted_clusters))
cluster_colors = {str(cluster): palette[i] for i, cluster in enumerate(sorted_clusters)}

# Create figure and subplots
fig, axes = plt.subplots(1, 2, figsize=(18, 9), gridspec_kw={'width_ratios': [1, 1]})

# Spatial plot
sns.scatterplot(x=spatial_coords['CenterX_global_px'], y=spatial_coords['CenterY_global_px'],
                hue=leiden_clusters, palette=cluster_colors, s=2, edgecolor=None, ax=axes[0])
axes[0].set_title('Leiden Clusters on Spatial Image')
axes[0].set_xlabel('CenterX (px)')
axes[0].set_ylabel('CenterY (px)')
axes[0].legend_.remove()  # Remove individual legend

# UMAP plot
sns.scatterplot(x=umap_coords[:, 0], y=umap_coords[:, 1],
                hue=leiden_clusters, palette=cluster_colors, s=5, edgecolor=None, ax=axes[1])
axes[1].set_title('Leiden Clusters on UMAP')
axes[1].set_xlabel('UMAP1')
axes[1].set_ylabel('UMAP2')
axes[1].legend_.remove()  # Remove individual legend

# Create a single shared legend closer to the plots
handles = [plt.Line2D([0], [0], marker='o', color=cluster_colors[str(cluster)], linestyle='', markersize=8)
           for cluster in sorted_clusters]
labels = [str(cluster) for cluster in sorted_clusters]
fig.legend(handles, labels, title="Leiden Clusters", bbox_to_anchor=(1, 0.5), loc='center left')

plt.tight_layout(rect=[0, 0, 0.85, 1])  # Adjust layout to fit legend closer to the plots
plt.show()
adata1_filtered

#300 minimum umi cutoff

In [ ]:
adata1_filtered_600 = adata1[(adata1.obs['nCount_RNA'] > 600) & (adata1.obs['nCount_RNA'] < 7500)].copy()
adata1_filtered_600

In [ ]:
#leiden_res_0.25

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Extract spatial and UMAP coordinates
spatial_coords = adata1_filtered_600.obs[['CenterX_global_px', 'CenterY_global_px']]
umap_coords = adata1_filtered_600.obsm['X_umap']
leiden_clusters = adata1_filtered_600.obs['leiden_res_0.25'].astype(str)  # Convert Leiden clusters to string for mapping

# Convert Leiden clusters to integers for proper numerical sorting
leiden_clusters_int = leiden_clusters.astype(int)
sorted_clusters = np.sort(leiden_clusters_int.unique())  # Sort numerically

# Define a consistent color palette
palette = sns.color_palette('tab20', len(sorted_clusters))
cluster_colors = {str(cluster): palette[i] for i, cluster in enumerate(sorted_clusters)}

# Create figure and subplots
fig, axes = plt.subplots(1, 2, figsize=(18, 9), gridspec_kw={'width_ratios': [1, 1]})

# Spatial plot
sns.scatterplot(x=spatial_coords['CenterX_global_px'], y=spatial_coords['CenterY_global_px'],
                hue=leiden_clusters, palette=cluster_colors, s=2, edgecolor=None, ax=axes[0])
axes[0].set_title('Leiden Clusters on Spatial Image')
axes[0].set_xlabel('CenterX (px)')
axes[0].set_ylabel('CenterY (px)')
axes[0].legend_.remove()  # Remove individual legend

# UMAP plot
sns.scatterplot(x=umap_coords[:, 0], y=umap_coords[:, 1],
                hue=leiden_clusters, palette=cluster_colors, s=5, edgecolor=None, ax=axes[1])
axes[1].set_title('Leiden Clusters on UMAP')
axes[1].set_xlabel('UMAP1')
axes[1].set_ylabel('UMAP2')
axes[1].legend_.remove()  # Remove individual legend

# Create a single shared legend closer to the plots
handles = [plt.Line2D([0], [0], marker='o', color=cluster_colors[str(cluster)], linestyle='', markersize=8)
           for cluster in sorted_clusters]
labels = [str(cluster) for cluster in sorted_clusters]
fig.legend(handles, labels, title="Leiden Clusters", bbox_to_anchor=(1, 0.5), loc='center left')

plt.tight_layout(rect=[0, 0, 0.85, 1])  # Adjust layout to fit legend closer to the plots
plt.show()

#600 minimum umi cutoff

In [ ]:
#will set filter thresholds to keep only cells with UMIs between:
#400-7500

adata1_filtered_400 = adata1[(adata1.obs['nCount_RNA'] > 400) & (adata1.obs['nCount_RNA'] < 7500)].copy()
adata1_filtered_400

In [ ]:
#leiden_res_0.25

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Extract spatial and UMAP coordinates
spatial_coords = adata1_filtered_400.obs[['CenterX_global_px', 'CenterY_global_px']]
umap_coords = adata1_filtered_400.obsm['X_umap']
leiden_clusters = adata1_filtered_400.obs['leiden_res_0.25'].astype(str)  # Convert Leiden clusters to string for mapping

# Convert Leiden clusters to integers for proper numerical sorting
leiden_clusters_int = leiden_clusters.astype(int)
sorted_clusters = np.sort(leiden_clusters_int.unique())  # Sort numerically

# Define a consistent color palette
palette = sns.color_palette('tab20', len(sorted_clusters))
cluster_colors = {str(cluster): palette[i] for i, cluster in enumerate(sorted_clusters)}

# Create figure and subplots
fig, axes = plt.subplots(1, 2, figsize=(18, 9), gridspec_kw={'width_ratios': [1, 1]})

# Spatial plot
sns.scatterplot(x=spatial_coords['CenterX_global_px'], y=spatial_coords['CenterY_global_px'],
                hue=leiden_clusters, palette=cluster_colors, s=2, edgecolor=None, ax=axes[0])
axes[0].set_title('Leiden Clusters on Spatial Image')
axes[0].set_xlabel('CenterX (px)')
axes[0].set_ylabel('CenterY (px)')
axes[0].legend_.remove()  # Remove individual legend

# UMAP plot
sns.scatterplot(x=umap_coords[:, 0], y=umap_coords[:, 1],
                hue=leiden_clusters, palette=cluster_colors, s=5, edgecolor=None, ax=axes[1])
axes[1].set_title('Leiden Clusters on UMAP')
axes[1].set_xlabel('UMAP1')
axes[1].set_ylabel('UMAP2')
axes[1].legend_.remove()  # Remove individual legend

# Create a single shared legend closer to the plots
handles = [plt.Line2D([0], [0], marker='o', color=cluster_colors[str(cluster)], linestyle='', markersize=8)
           for cluster in sorted_clusters]
labels = [str(cluster) for cluster in sorted_clusters]
fig.legend(handles, labels, title="Leiden Clusters", bbox_to_anchor=(1, 0.5), loc='center left')

plt.tight_layout(rect=[0, 0, 0.85, 1])  # Adjust layout to fit legend closer to the plots
plt.show()
adata1_filtered

#400 minimum umi cutoff

In [ ]:
#will set filter thresholds to keep only cells with UMIs between:
#500-7500

adata1_filtered_500 = adata1[(adata1.obs['nCount_RNA'] > 500) & (adata1.obs['nCount_RNA'] < 7500)].copy()
adata1_filtered_500

In [ ]:
#leiden_res_0.25

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Extract spatial and UMAP coordinates
spatial_coords = adata1_filtered_500.obs[['CenterX_global_px', 'CenterY_global_px']]
umap_coords = adata1_filtered_500.obsm['X_umap']
leiden_clusters = adata1_filtered_500.obs['leiden_res_0.25'].astype(str)  # Convert Leiden clusters to string for mapping

# Convert Leiden clusters to integers for proper numerical sorting
leiden_clusters_int = leiden_clusters.astype(int)
sorted_clusters = np.sort(leiden_clusters_int.unique())  # Sort numerically

# Define a consistent color palette
palette = sns.color_palette('tab20', len(sorted_clusters))
cluster_colors = {str(cluster): palette[i] for i, cluster in enumerate(sorted_clusters)}

# Create figure and subplots
fig, axes = plt.subplots(1, 2, figsize=(18, 9), gridspec_kw={'width_ratios': [1, 1]})

# Spatial plot
sns.scatterplot(x=spatial_coords['CenterX_global_px'], y=spatial_coords['CenterY_global_px'],
                hue=leiden_clusters, palette=cluster_colors, s=2, edgecolor=None, ax=axes[0])
axes[0].set_title('Leiden Clusters on Spatial Image')
axes[0].set_xlabel('CenterX (px)')
axes[0].set_ylabel('CenterY (px)')
axes[0].legend_.remove()  # Remove individual legend

# UMAP plot
sns.scatterplot(x=umap_coords[:, 0], y=umap_coords[:, 1],
                hue=leiden_clusters, palette=cluster_colors, s=5, edgecolor=None, ax=axes[1])
axes[1].set_title('Leiden Clusters on UMAP')
axes[1].set_xlabel('UMAP1')
axes[1].set_ylabel('UMAP2')
axes[1].legend_.remove()  # Remove individual legend

# Create a single shared legend closer to the plots
handles = [plt.Line2D([0], [0], marker='o', color=cluster_colors[str(cluster)], linestyle='', markersize=8)
           for cluster in sorted_clusters]
labels = [str(cluster) for cluster in sorted_clusters]
fig.legend(handles, labels, title="Leiden Clusters", bbox_to_anchor=(1, 0.5), loc='center left')

plt.tight_layout(rect=[0, 0, 0.85, 1])  # Adjust layout to fit legend closer to the plots
plt.show()
adata1_filtered

#500 minimum umi cutoff

In [ ]:
#will set filter thresholds to keep only cells with UMIs between:
#800-7500

adata1_filtered_800 = adata1[(adata1.obs['nCount_RNA'] > 800) & (adata1.obs['nCount_RNA'] < 7500)].copy()
adata1_filtered_800

In [ ]:
#leiden_res_0.25

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Extract spatial and UMAP coordinates
spatial_coords = adata1_filtered_800.obs[['CenterX_global_px', 'CenterY_global_px']]
umap_coords = adata1_filtered_800.obsm['X_umap']
leiden_clusters = adata1_filtered_800.obs['leiden_res_0.25'].astype(str)  # Convert Leiden clusters to string for mapping

# Convert Leiden clusters to integers for proper numerical sorting
leiden_clusters_int = leiden_clusters.astype(int)
sorted_clusters = np.sort(leiden_clusters_int.unique())  # Sort numerically

# Define a consistent color palette
palette = sns.color_palette('tab20', len(sorted_clusters))
cluster_colors = {str(cluster): palette[i] for i, cluster in enumerate(sorted_clusters)}

# Create figure and subplots
fig, axes = plt.subplots(1, 2, figsize=(18, 9), gridspec_kw={'width_ratios': [1, 1]})

# Spatial plot
sns.scatterplot(x=spatial_coords['CenterX_global_px'], y=spatial_coords['CenterY_global_px'],
                hue=leiden_clusters, palette=cluster_colors, s=2, edgecolor=None, ax=axes[0])
axes[0].set_title('Leiden Clusters on Spatial Image')
axes[0].set_xlabel('CenterX (px)')
axes[0].set_ylabel('CenterY (px)')
axes[0].legend_.remove()  # Remove individual legend

# UMAP plot
sns.scatterplot(x=umap_coords[:, 0], y=umap_coords[:, 1],
                hue=leiden_clusters, palette=cluster_colors, s=5, edgecolor=None, ax=axes[1])
axes[1].set_title('Leiden Clusters on UMAP')
axes[1].set_xlabel('UMAP1')
axes[1].set_ylabel('UMAP2')
axes[1].legend_.remove()  # Remove individual legend

# Create a single shared legend closer to the plots
handles = [plt.Line2D([0], [0], marker='o', color=cluster_colors[str(cluster)], linestyle='', markersize=8)
           for cluster in sorted_clusters]
labels = [str(cluster) for cluster in sorted_clusters]
fig.legend(handles, labels, title="Leiden Clusters", bbox_to_anchor=(1, 0.5), loc='center left')

plt.tight_layout(rect=[0, 0, 0.85, 1])  # Adjust layout to fit legend closer to the plots
plt.show()
adata1_filtered

#800 minimum umi cutoff

In [ ]:
#will set filter thresholds to keep only cells with UMIs between:
#1000-7500

adata1_filtered_1000 = adata1[(adata1.obs['nCount_RNA'] > 1000) & (adata1.obs['nCount_RNA'] < 7500)].copy()
adata1_filtered_1000

In [ ]:
#leiden_res_0.25

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Extract spatial and UMAP coordinates
spatial_coords = adata1_filtered_1000.obs[['CenterX_global_px', 'CenterY_global_px']]
umap_coords = adata1_filtered_1000.obsm['X_umap']
leiden_clusters = adata1_filtered_1000.obs['leiden_res_0.25'].astype(str)  # Convert Leiden clusters to string for mapping

# Convert Leiden clusters to integers for proper numerical sorting
leiden_clusters_int = leiden_clusters.astype(int)
sorted_clusters = np.sort(leiden_clusters_int.unique())  # Sort numerically

# Define a consistent color palette
palette = sns.color_palette('tab20', len(sorted_clusters))
cluster_colors = {str(cluster): palette[i] for i, cluster in enumerate(sorted_clusters)}

# Create figure and subplots
fig, axes = plt.subplots(1, 2, figsize=(18, 9), gridspec_kw={'width_ratios': [1, 1]})

# Spatial plot
sns.scatterplot(x=spatial_coords['CenterX_global_px'], y=spatial_coords['CenterY_global_px'],
                hue=leiden_clusters, palette=cluster_colors, s=2, edgecolor=None, ax=axes[0])
axes[0].set_title('Leiden Clusters on Spatial Image')
axes[0].set_xlabel('CenterX (px)')
axes[0].set_ylabel('CenterY (px)')
axes[0].legend_.remove()  # Remove individual legend

# UMAP plot
sns.scatterplot(x=umap_coords[:, 0], y=umap_coords[:, 1],
                hue=leiden_clusters, palette=cluster_colors, s=5, edgecolor=None, ax=axes[1])
axes[1].set_title('Leiden Clusters on UMAP')
axes[1].set_xlabel('UMAP1')
axes[1].set_ylabel('UMAP2')
axes[1].legend_.remove()  # Remove individual legend

# Create a single shared legend closer to the plots
handles = [plt.Line2D([0], [0], marker='o', color=cluster_colors[str(cluster)], linestyle='', markersize=8)
           for cluster in sorted_clusters]
labels = [str(cluster) for cluster in sorted_clusters]
fig.legend(handles, labels, title="Leiden Clusters", bbox_to_anchor=(1, 0.5), loc='center left')

plt.tight_layout(rect=[0, 0, 0.85, 1])  # Adjust layout to fit legend closer to the plots
plt.show()
adata1_filtered

#1000 minimum umi cutoff

In [ ]:
#will set filter thresholds to keep only cells with UMIs between:
#1500-7500

adata1_filtered_1500 = adata1[(adata1.obs['nCount_RNA'] > 1500) & (adata1.obs['nCount_RNA'] < 7500)].copy()
adata1_filtered_1500

In [ ]:
#leiden_res_0.25

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Extract spatial and UMAP coordinates
spatial_coords = adata1_filtered_1500.obs[['CenterX_global_px', 'CenterY_global_px']]
umap_coords = adata1_filtered_1500.obsm['X_umap']
leiden_clusters = adata1_filtered_1500.obs['leiden_res_0.25'].astype(str)  # Convert Leiden clusters to string for mapping

# Convert Leiden clusters to integers for proper numerical sorting
leiden_clusters_int = leiden_clusters.astype(int)
sorted_clusters = np.sort(leiden_clusters_int.unique())  # Sort numerically

# Define a consistent color palette
palette = sns.color_palette('tab20', len(sorted_clusters))
cluster_colors = {str(cluster): palette[i] for i, cluster in enumerate(sorted_clusters)}

# Create figure and subplots
fig, axes = plt.subplots(1, 2, figsize=(18, 9), gridspec_kw={'width_ratios': [1, 1]})

# Spatial plot
sns.scatterplot(x=spatial_coords['CenterX_global_px'], y=spatial_coords['CenterY_global_px'],
                hue=leiden_clusters, palette=cluster_colors, s=2, edgecolor=None, ax=axes[0])
axes[0].set_title('Leiden Clusters on Spatial Image')
axes[0].set_xlabel('CenterX (px)')
axes[0].set_ylabel('CenterY (px)')
axes[0].legend_.remove()  # Remove individual legend

# UMAP plot
sns.scatterplot(x=umap_coords[:, 0], y=umap_coords[:, 1],
                hue=leiden_clusters, palette=cluster_colors, s=5, edgecolor=None, ax=axes[1])
axes[1].set_title('Leiden Clusters on UMAP')
axes[1].set_xlabel('UMAP1')
axes[1].set_ylabel('UMAP2')
axes[1].legend_.remove()  # Remove individual legend

# Create a single shared legend closer to the plots
handles = [plt.Line2D([0], [0], marker='o', color=cluster_colors[str(cluster)], linestyle='', markersize=8)
           for cluster in sorted_clusters]
labels = [str(cluster) for cluster in sorted_clusters]
fig.legend(handles, labels, title="Leiden Clusters", bbox_to_anchor=(1, 0.5), loc='center left')

plt.tight_layout(rect=[0, 0, 0.85, 1])  # Adjust layout to fit legend closer to the plots
plt.show()
adata1_filtered

#1500 minimum umi cutoff

In [ ]:
#after filtering

adata1_filtered_600.layers["counts"] = adata1_filtered_600.X.copy()
sc.pp.normalize_total(adata1_filtered_600, inplace=True)
sc.pp.log1p(adata1_filtered_600)
sc.pp.pca(adata1_filtered_600)
sc.pp.neighbors(adata1_filtered_600)

In [ ]:
sc.tl.umap(adata1_filtered_600)
sc.tl.leiden(adata1_filtered_600)

In [ ]:
adata1_filtered_600

In [ ]:
#leiden_res_0.25

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Extract spatial and UMAP coordinates
spatial_coords = adata1_filtered_600.obs[['CenterX_global_px', 'CenterY_global_px']]
umap_coords = adata1_filtered_600.obsm['X_umap']
leiden_clusters = adata1_filtered_600.obs['leiden_res_0.25'].astype(str)  # Convert Leiden clusters to string for mapping

# Convert Leiden clusters to integers for proper numerical sorting
leiden_clusters_int = leiden_clusters.astype(int)
sorted_clusters = np.sort(leiden_clusters_int.unique())  # Sort numerically

# Define a consistent color palette
palette = sns.color_palette('tab20', len(sorted_clusters))
cluster_colors = {str(cluster): palette[i] for i, cluster in enumerate(sorted_clusters)}

# Create figure and subplots
fig, axes = plt.subplots(1, 2, figsize=(18, 9), gridspec_kw={'width_ratios': [1, 1]})

# Spatial plot
sns.scatterplot(x=spatial_coords['CenterX_global_px'], y=spatial_coords['CenterY_global_px'],
                hue=leiden_clusters, palette=cluster_colors, s=2, edgecolor=None, ax=axes[0])
axes[0].set_title('Leiden Clusters on Spatial Image')
axes[0].set_xlabel('CenterX (px)')
axes[0].set_ylabel('CenterY (px)')
axes[0].legend_.remove()  # Remove individual legend

# UMAP plot
sns.scatterplot(x=umap_coords[:, 0], y=umap_coords[:, 1],
                hue=leiden_clusters, palette=cluster_colors, s=5, edgecolor=None, ax=axes[1])
axes[1].set_title('Leiden Clusters on UMAP')
axes[1].set_xlabel('UMAP1')
axes[1].set_ylabel('UMAP2')
axes[1].legend_.remove()  # Remove individual legend

# Create a single shared legend closer to the plots
handles = [plt.Line2D([0], [0], marker='o', color=cluster_colors[str(cluster)], linestyle='', markersize=8)
           for cluster in sorted_clusters]
labels = [str(cluster) for cluster in sorted_clusters]
fig.legend(handles, labels, title="Leiden Clusters", bbox_to_anchor=(1, 0.5), loc='center left')

plt.tight_layout(rect=[0, 0, 0.85, 1])  # Adjust layout to fit legend closer to the plots
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

adata = adata1_filtered_600

# Leiden clusters
clusters = adata.obs['leiden'].astype(str)
unique_clusters = np.sort(clusters.unique())

# Color handling
if 'leiden_colors' in adata.uns:
    color_list = adata.uns['leiden_colors']
    cluster_colors = dict(zip(adata.obs['leiden'].cat.categories.astype(str), color_list))
    print("Using pre-defined Scanpy leiden colors")
else:
    palette = sns.color_palette('tab20', len(unique_clusters))
    cluster_colors = dict(zip(unique_clusters, palette))
    print("Using generated tab20 palette")

default_color = 'lightgray'  # string name is fine for matplotlib

# Coordinates
spatial_coords = adata.obsm['spatial']
x_spatial, y_spatial = spatial_coords[:, 0], spatial_coords[:, 1]
umap_coords = adata.obsm['X_umap']

# Loop over clusters
for clust in unique_clusters:
    highlight_color = cluster_colors.get(clust, 'red')  # this is either tuple or will fallback to named color

    # Build color array using list comprehension (safest way)
    colors_spatial = [
        highlight_color if c == clust else default_color
        for c in clusters
    ]
    colors_umap = colors_spatial  # same for UMAP

    fig, axes = plt.subplots(1, 2, figsize=(18, 8))

    # Spatial plot
    axes[0].scatter(x_spatial, y_spatial, c=colors_spatial, s=2, edgecolor='none')
    axes[0].set_title(f'Spatial: Leiden {clust}', fontsize=14)
    axes[0].set_xlabel('X (px)')
    axes[0].set_ylabel('Y (px)')
    axes[0].invert_yaxis()
    axes[0].set_aspect('equal')

    # UMAP plot
    axes[1].scatter(umap_coords[:, 0], umap_coords[:, 1], c=colors_umap, s=5, edgecolor='none')
    axes[1].set_title(f'UMAP: Leiden {clust}', fontsize=14)
    axes[1].set_xlabel('UMAP 1')
    axes[1].set_ylabel('UMAP 2')

    # Legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', label=f'Cluster {clust}',
               markerfacecolor=highlight_color, markersize=10, markeredgewidth=0),
        Line2D([0], [0], marker='o', color='w', label='Other cells',
               markerfacecolor=default_color, markersize=10, markeredgewidth=0)
    ]
    axes[1].legend(handles=legend_elements, loc='upper right')

    plt.suptitle(f'Leiden Cluster: {clust}', fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
# Define the Leiden clusters you want to keep
clusters_to_keep = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14]

# Make sure the column is string or category for safe comparison
leiden_series = adata1_filtered_600.obs['leiden'].astype(str)

# Create a boolean mask for cells in the desired clusters
mask = leiden_series.isin([str(c) for c in clusters_to_keep])

# Check how many cells we're keeping
print(f"Original cells: {adata1_filtered_600.n_obs}")
print(f"Cells after filtering: {mask.sum()}")
print(f"Removed cells: {adata1_filtered_600.n_obs - mask.sum()}")

# Create the new filtered AnnData object
adata_filtered_600_leiden_filtered = adata1_filtered_600[mask].copy()

# Optional: Check the remaining clusters to confirm
print("\nRemaining Leiden clusters:")
print(adata_filtered_600_leiden_filtered.obs['leiden'].value_counts().sort_index())

# Now you can use this new object
adata_filtered_600_leiden_filtered

In [ ]:
#after filtering again

adata_filtered_600_leiden_filtered.layers["counts"] = adata_filtered_600_leiden_filtered.X.copy()
sc.pp.normalize_total(adata_filtered_600_leiden_filtered, inplace=True)
sc.pp.log1p(adata_filtered_600_leiden_filtered)
sc.pp.pca(adata_filtered_600_leiden_filtered)
sc.pp.neighbors(adata_filtered_600_leiden_filtered)

In [ ]:
sc.tl.umap(adata_filtered_600_leiden_filtered)
sc.tl.leiden(adata_filtered_600_leiden_filtered)

adata_filtered_600_leiden_filtered

In [ ]:
#leiden

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Extract spatial and UMAP coordinates
spatial_coords = adata_filtered_600_leiden_filtered.obs[['CenterX_global_px', 'CenterY_global_px']]
umap_coords = adata_filtered_600_leiden_filtered.obsm['X_umap']
leiden_clusters = adata_filtered_600_leiden_filtered.obs['leiden'].astype(str)  # Convert Leiden clusters to string for mapping

# Convert Leiden clusters to integers for proper numerical sorting
leiden_clusters_int = leiden_clusters.astype(int)
sorted_clusters = np.sort(leiden_clusters_int.unique())  # Sort numerically

# Define a consistent color palette
palette = sns.color_palette('tab20', len(sorted_clusters))
cluster_colors = {str(cluster): palette[i] for i, cluster in enumerate(sorted_clusters)}

# Create figure and subplots
fig, axes = plt.subplots(1, 2, figsize=(18, 9), gridspec_kw={'width_ratios': [1, 1]})

# Spatial plot
sns.scatterplot(x=spatial_coords['CenterX_global_px'], y=spatial_coords['CenterY_global_px'],
                hue=leiden_clusters, palette=cluster_colors, s=2, edgecolor=None, ax=axes[0])
axes[0].set_title('Leiden Clusters on Spatial Image')
axes[0].set_xlabel('CenterX (px)')
axes[0].set_ylabel('CenterY (px)')
axes[0].legend_.remove()  # Remove individual legend

# UMAP plot
sns.scatterplot(x=umap_coords[:, 0], y=umap_coords[:, 1],
                hue=leiden_clusters, palette=cluster_colors, s=5, edgecolor=None, ax=axes[1])
axes[1].set_title('Leiden Clusters on UMAP')
axes[1].set_xlabel('UMAP1')
axes[1].set_ylabel('UMAP2')
axes[1].legend_.remove()  # Remove individual legend

# Create a single shared legend closer to the plots
handles = [plt.Line2D([0], [0], marker='o', color=cluster_colors[str(cluster)], linestyle='', markersize=8)
           for cluster in sorted_clusters]
labels = [str(cluster) for cluster in sorted_clusters]
fig.legend(handles, labels, title="Leiden Clusters", bbox_to_anchor=(1, 0.5), loc='center left')

plt.tight_layout(rect=[0, 0, 0.85, 1])  # Adjust layout to fit legend closer to the plots
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

adata = adata_filtered_600_leiden_filtered

# Leiden clusters
clusters = adata.obs['leiden'].astype(str)
unique_clusters = np.sort(clusters.unique())

# Color handling
if 'leiden_colors' in adata.uns:
    color_list = adata.uns['leiden_colors']
    cluster_colors = dict(zip(adata.obs['leiden'].cat.categories.astype(str), color_list))
    print("Using pre-defined Scanpy leiden colors")
else:
    palette = sns.color_palette('tab20', len(unique_clusters))
    cluster_colors = dict(zip(unique_clusters, palette))
    print("Using generated tab20 palette")

default_color = 'lightgray'  # string name is fine for matplotlib

# Coordinates
spatial_coords = adata.obsm['spatial']
x_spatial, y_spatial = spatial_coords[:, 0], spatial_coords[:, 1]
umap_coords = adata.obsm['X_umap']

# Loop over clusters
for clust in unique_clusters:
    highlight_color = cluster_colors.get(clust, 'red')  # this is either tuple or will fallback to named color

    # Build color array using list comprehension (safest way)
    colors_spatial = [
        highlight_color if c == clust else default_color
        for c in clusters
    ]
    colors_umap = colors_spatial  # same for UMAP

    fig, axes = plt.subplots(1, 2, figsize=(18, 8))

    # Spatial plot
    axes[0].scatter(x_spatial, y_spatial, c=colors_spatial, s=2, edgecolor='none')
    axes[0].set_title(f'Spatial: Leiden {clust}', fontsize=14)
    axes[0].set_xlabel('X (px)')
    axes[0].set_ylabel('Y (px)')
    axes[0].invert_yaxis()
    axes[0].set_aspect('equal')

    # UMAP plot
    axes[1].scatter(umap_coords[:, 0], umap_coords[:, 1], c=colors_umap, s=5, edgecolor='none')
    axes[1].set_title(f'UMAP: Leiden {clust}', fontsize=14)
    axes[1].set_xlabel('UMAP 1')
    axes[1].set_ylabel('UMAP 2')

    # Legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', label=f'Cluster {clust}',
               markerfacecolor=highlight_color, markersize=10, markeredgewidth=0),
        Line2D([0], [0], marker='o', color='w', label='Other cells',
               markerfacecolor=default_color, markersize=10, markeredgewidth=0)
    ]
    axes[1].legend(handles=legend_elements, loc='upper right')

    plt.suptitle(f'Leiden Cluster: {clust}', fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
import scanpy as sc

# 1. PCA + neighbors (only if not already done)
#sc.pp.pca(adata1, n_comps=50, svd_solver='arpack')
#sc.pp.neighbors(adata1, n_neighbors=30, n_pcs=40)

# 2. Multi-resolution Leiden - correct syntax for Scanpy ≥1.9.6 + leidenalg ≥0.10
resolutions = [0.1, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.2, 1.5, 2.0, 4.0, 8.0]

for res in resolutions:
    print(f"Running Leiden at resolution {res} ...")

    sc.tl.leiden(
        adata_filtered_600_leiden_filtered,
        resolution=res,
        key_added=f"leiden_res_{res}",      #  leiden_res_0.25, leiden_res_0.5, etc.
        random_state=42,                    # for perfect reproducibility
        n_iterations=2                      # 2 is almost always enough
        # flavor and directed are now ignored - the default is igraph + undirected
    )

    n_clusters = adata_filtered_600_leiden_filtered.obs[f"leiden_res_{res}"].nunique()
    print(f"    {n_clusters} clusters")

# 3. Make resolution 1.0 the default column
# adata_filtered_600_leiden_filtered.obs['leiden'] = adata_filtered_600_leiden_filtered.obs['leiden_res_1.0'].astype('category')

# 4. Confirm success
print("\nSuccess! These columns are now in adata_filtered_600_leiden_filtered.obs:")
print([c for c in adata_filtered_600_leiden_filtered.obs.columns if c.startswith('leiden_res_')])
print("Default column: 'leiden'")

In [ ]:
import os

outdir = "/u/project/cluo/mbaig/cosmx/analyses/20251130_MB_basal_3/processed_data/20260106/"
os.makedirs(outdir, exist_ok=True)

# Save AnnData objects
adata_filtered_600_leiden_filtered.write(os.path.join(outdir, "20260107_bgs5_600_umi_filtered_normalized.h5ad"))